In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

from tensorflow import keras
import tensorflow as tf

from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:1024"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

2026-09-15 07:35:43.920941: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    """Walk upward from `start` to find the directory containing this repo's data/
    folder, so this notebook works regardless of the kernel's working directory
    (VS Code defaults to the notebook's own folder; classic Jupyter defaults to
    wherever you launched it from)."""
    for candidate in [start.resolve()] + list(start.resolve().parents):
        if (candidate / "data" / "labels" / "mb24").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repo root (a data/labels/mb24 directory) above "
        f"{start}. Make sure data.tar.gz has been extracted per the main README."
    )

REPO_ROOT = _find_repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "data"
print(f"Using DATA_DIR = {DATA_DIR}")


Using DATA_DIR = /workspace/LFreeDA/data


## Useful functions 

In [3]:
import warnings
from concurrent.futures import ThreadPoolExecutor

# Some malware-derived and benign images legitimately exceed Pillow's default
# decompression-bomb pixel-count threshold. Disable the check globally (once,
# for both loaders below) instead of per-call, and silence the corresponding
# warning so it doesn't spam stdout during normal loading.
Image.MAX_IMAGE_PIXELS = None
warnings.filterwarnings("ignore", category=Image.DecompressionBombWarning)

def change_attack_label(x):
    label = [1.]
    return label

def _load_one_malware_image(args):
    filename, image_path = args
    hash_id = filename.split(".")[0]
    f = os.path.join(image_path, filename)
    image = Image.open(f).convert('RGB')
    image = image.resize((56, 56), Image.LANCZOS)
    image = np.array(image, dtype=int)
    return hash_id, image, image_path + "/" + filename

def load_image_malware(image_path, label_path, max_workers=None):
    labels = pd.read_csv(label_path, header=0)
    # O(1) label lookup instead of re-scanning the whole dataframe per image
    label_map = dict(zip(labels["malware SHA-256"], labels["Label"]))

    filenames = [
        filename for filename in os.listdir(image_path)
        if filename.endswith(".png") and filename.split(".")[0] in label_map
    ]

    # Image decode/resize is I/O- and PIL-bound (PIL releases the GIL for
    # most of this work), so a thread pool parallelizes it well without the
    # process-pool overhead of pickling images back to the main process.
    max_workers = max_workers or os.cpu_count()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(executor.map(
            lambda fn: _load_one_malware_image((fn, image_path)), filenames
        ))

    x = [image for _, image, _ in results]
    y = [[label_map[hash_id]] for hash_id, _, _ in results]
    paths = [p for _, _, p in results]

    x = np.asarray(x)
    y = np.asarray(y)
    x = x.astype('float32') / 255.
    paths = np.array(paths, dtype=object)

    return x, y, paths

def _load_one_normal_image(args):
    filename, directory_path, image_size_limit = args
    file_path = os.path.join(directory_path, filename)
    try:
        with Image.open(file_path) as img:
            # Check if the image size is within the allowed limit
            if img.width * img.height <= image_size_limit:
                img = img.convert('RGB')
                img = img.resize((56, 56), Image.LANCZOS)
                return np.array(img, dtype=int), directory_path + "/" + filename
            else:
                # Silently skip oversized images (still skipped, just not logged).
                return None
    except (Image.DecompressionBombError, OSError) as e:
        print(f"Error loading image {filename}: {e}")
        return None

def load_image_normal(directory_path, max_workers=None):
    image_size_limit = 178956970  # Maximum allowed pixels per image

    filenames = [
        filename for filename in os.listdir(directory_path)
        if filename.endswith(".jpg") or filename.endswith(".png") or filename.endswith(".jpeg")
    ]

    max_workers = max_workers or os.cpu_count()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(executor.map(
            lambda fn: _load_one_normal_image((fn, directory_path, image_size_limit)), filenames
        ))

    results = [r for r in results if r is not None]
    image_list = [img for img, _ in results]
    paths = [p for _, p in results]

    image_list = np.asarray(image_list)
    image_list = image_list.astype('float32') / 255.
    paths = np.array(paths, dtype=object)

    return image_list, paths


## Load malware data

### march

In [4]:
label_path = str(DATA_DIR / "labels/mb24/March/march_malware.csv")
img_path = str(DATA_DIR / "image_features/mb24/march/march")
malware_march_x, malware_march_y, malware_march_path=load_image_malware(img_path, label_path)

### april

In [5]:
label_path = str(DATA_DIR / "labels/mb24/April/april_malware.csv")
img_path = str(DATA_DIR / "image_features/mb24/april/april")
malware_april_x, malware_april_y,malware_april_path =load_image_malware(img_path, label_path)

### may

In [6]:
label_path = str(DATA_DIR / "labels/mb24/May/may_malware.csv")
img_path = str(DATA_DIR / "image_features/mb24/may/may")
malware_may_x, malware_may_y, malware_may_path =load_image_malware(img_path, label_path)

### oct

In [7]:
label_path = str(DATA_DIR / "labels/mb24/Oct/oct_malware.csv")
img_path = str(DATA_DIR / "image_features/mb24/oct")
malware_oct_x, malware_oct_y, malware_oct_path =load_image_malware(img_path, label_path)

### nov

In [8]:
label_path = str(DATA_DIR / "labels/mb24/Nov/nov_malware.csv")
img_path = str(DATA_DIR / "image_features/mb24/nov")
malware_nov_x, malware_nov_y, malware_nov_path =load_image_malware(img_path, label_path)

## Load normal data source

In [9]:
img_path = str(DATA_DIR / "image_features/benign_source/dataset1")
source_normal_x_1, source_normal_path_1 = load_image_normal(img_path)

In [10]:
source_normal_y_1 = np.zeros((source_normal_x_1.shape[0],1))

In [11]:
img_path = str(DATA_DIR / "image_features/benign_source/dataset2")
source_normal_x_2, source_normal_path_2 = load_image_normal(img_path)

In [12]:
source_normal_y_2 = np.zeros((source_normal_x_2.shape[0],1))

In [13]:
img_path = str(DATA_DIR / "image_features/benign_source/dataset3")
source_normal_x_3, source_normal_path_3 = load_image_normal(img_path)

In [14]:
source_normal_y_3 = np.zeros((source_normal_x_3.shape[0],1))

In [15]:
img_path = str(DATA_DIR / "image_features/benign_source/dataset4")
source_normal_x_4, source_normal_path_4 = load_image_normal(img_path)

In [16]:
source_normal_y_4 = np.zeros((source_normal_x_4.shape[0],1))

### merge

In [17]:
source_normal_x = np.concatenate((source_normal_x_1, source_normal_x_2, source_normal_x_3, source_normal_x_4), axis = 0)
source_normal_y = np.concatenate((source_normal_y_1, source_normal_y_2, source_normal_y_3, source_normal_y_4), axis = 0)
source_normal_path = np.concatenate((source_normal_path_1, source_normal_path_2, source_normal_path_3, source_normal_path_4), axis = 0)

## Load normal data target

In [18]:
img_path = str(DATA_DIR / "image_features/benign_target/dataset1")
target_normal_x_1, target_normal_path_1 = load_image_normal(img_path)

In [19]:
target_normal_y_1 = np.zeros((target_normal_x_1.shape[0],1))

In [20]:
img_path = str(DATA_DIR / "image_features/benign_target/dataset2")
target_normal_x_2, target_normal_path_2 = load_image_normal(img_path)

In [21]:
target_normal_y_2 = np.zeros((target_normal_x_2.shape[0],1))

In [22]:
img_path = str(DATA_DIR / "image_features/benign_target/dataset3")
target_normal_x_3, target_normal_path_3 = load_image_normal(img_path)

In [23]:
target_normal_y_3 = np.zeros((target_normal_x_3.shape[0],1))

In [24]:
img_path = str(DATA_DIR / "image_features/benign_target/dataset4")
target_normal_x_4, target_normal_path_4 = load_image_normal(img_path)

In [25]:
target_normal_y_4 = np.zeros((target_normal_x_4.shape[0],1))

### merge

In [26]:
target_normal_x = np.concatenate((target_normal_x_1, target_normal_x_2, target_normal_x_3, target_normal_x_4), axis = 0)
target_normal_y = np.concatenate((target_normal_y_1, target_normal_y_2, target_normal_y_3, target_normal_y_4), axis = 0)
target_normal_path = np.concatenate((target_normal_path_1, target_normal_path_2, target_normal_path_3, target_normal_path_4), axis = 0)

## Nov

In [27]:
source_malware_x = np.concatenate((malware_march_x, malware_april_x, malware_may_x), axis = 0)
source_malware_y = np.concatenate((malware_march_y, malware_april_y, malware_may_y), axis = 0)
source_malware_path = np.concatenate((malware_march_path, malware_april_path, malware_may_path), axis = 0)

target_malware_train_x = malware_oct_x
target_malware_train_y = malware_oct_y
target_malware_train_path = malware_oct_path

target_malware_test_x = malware_nov_x
target_malware_test_y = malware_nov_y
target_malware_test_path = malware_nov_path

target_normal_train_x, target_normal_test_x, \
target_normal_train_y, target_normal_test_y, \
target_normal_train_path, target_normal_test_path = train_test_split(target_normal_x, target_normal_y, target_normal_path, test_size=0.5, random_state=42)

source_malware_y = np.apply_along_axis(change_attack_label, 1, source_malware_y)
target_malware_train_y = np.apply_along_axis(change_attack_label, 1, target_malware_train_y)
target_malware_test_y = np.apply_along_axis(change_attack_label, 1, target_malware_test_y)

source_x = np.concatenate((source_malware_x, source_normal_x), axis = 0)
source_y = np.concatenate((source_malware_y, source_normal_y), axis = 0)
source_path = np.concatenate((source_malware_path, source_normal_path), axis = 0)

target_x_train = np.concatenate((target_malware_train_x, target_normal_train_x), axis = 0)
target_y_train = np.concatenate((target_malware_train_y, target_normal_train_y), axis = 0)
target_path_train = np.concatenate((target_malware_train_path, target_normal_train_path), axis = 0)

target_x_test = np.concatenate((target_malware_test_x, target_normal_test_x), axis = 0)
target_y_test = np.concatenate((target_malware_test_y, target_normal_test_y), axis = 0)
target_path_test = np.concatenate((target_malware_test_path, target_normal_test_path), axis = 0)

#one-hot encode labels

source_y = tf.keras.utils.to_categorical(source_y, num_classes = 2)
target_y_train  = tf.keras.utils.to_categorical(target_y_train, num_classes = 2)
target_y_test = tf.keras.utils.to_categorical(target_y_test , num_classes = 2)

source_x_train, source_x_test, \
source_y_train, source_y_test, \
source_path_train, source_path_test= train_test_split(source_x, source_y, source_path, test_size=0.25, random_state=42)



### Load MaxDIRep

In [28]:
# …later, or in a new script/session…
generator = keras.models.load_model(str(DATA_DIR / "stepI_trained_models/mb24/nov/generator"))
classifier = keras.models.load_model(str(DATA_DIR / "stepI_trained_models/mb24/nov/classifier"))


y_target_class_pred = classifier.predict(generator(target_x_train)).argmax(1)

# 1. Original noisy‐label accuracy on training data
y_noisy = y_target_class_pred  
y_true_all = target_y_train.argmax(axis=1)
acc_orig = accuracy_score(y_true_all, y_target_class_pred)
print(f"Original noisy accuracy on target train: {acc_orig:.2%} "
      f"(on {len(y_true_all)} samples)")

# target_y_test = target_y_test.argmax(axis=1)
# source_y = source_y.argmax(axis=1)

2026-09-15 07:36:40.326374: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-15 07:36:40.712923: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1532] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21995 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9


2026-09-15 07:36:41.709326: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8100


189/189 [==============================] - 0s 482us/step
Original noisy accuracy on target train: 82.27% (on 6019 samples)


2026-09-15 07:36:42.792474: I tensorflow/stream_executor/cuda/cuda_blas.cc:1786] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


In [29]:
# target_x_train:  (N, …)
Z_maps = generator.predict(target_x_train)        # shape = (N, 12, 12, 64)
N = Z_maps.shape[0]

Z = Z_maps.reshape(N, -1)    
 


189/189 [==============================] - 0s 590us/step


### local outlier factor

In [30]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import accuracy_score

# 1. PCA → 50 dims (speed + regularization)
pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
Z_reduced = pca.fit_transform(Z)  # (N, 50)

# 2. Softmax probabilities + confidence
probs = classifier.predict(Z)  # shape = (N, num_classes)
conf = np.max(probs, axis=1)           # best-class probability

# 3. Build masks: LOF-only, confidence-only, and combined
keep_lof   = np.zeros(N, dtype=bool)
keep_conf  = np.zeros(N, dtype=bool)
keep_combo = np.zeros(N, dtype=bool)
lof_contam = 0.2 # fraction of outliers per class
conf_thresh = 0.95  # confidence cutoff

#y_noisy = y_target_class_pred
# 4. Per-class LOF filtering + confidence gating
for c in np.unique(y_noisy):
    idx = np.where(y_noisy == c)[0]
    Zc = Z_reduced[idx]

    # fit LOF on class-c embeddings
    lof = LocalOutlierFactor(n_neighbors=50, contamination=lof_contam)
    preds = lof.fit_predict(Zc)   # +1=inlier, -1=outlier
    inliers = preds == 1

    # update LOF-only mask
    keep_lof[idx[inliers]] = True

    # update confidence-only mask
    conf_mask = conf[idx] >= conf_thresh
    keep_conf[idx[conf_mask]] = True

    # update combined mask
    keep_combo[idx[inliers & conf_mask]] = True

# 5. Slice out subsets
y_true_conf  = y_true_all[keep_conf]
y_pred_conf  = y_noisy[keep_conf]

y_true_combo = y_true_all[keep_combo]
y_pred_combo = y_noisy[keep_combo]

# 6. Compute & print accuracies
acc_orig  = accuracy_score(y_true_all, y_noisy)
acc_conf  = accuracy_score(y_true_conf, y_pred_conf)
acc_combo = accuracy_score(y_true_combo, y_pred_combo)

print(f"Original accuracy             : {acc_orig:.2%} on {N} samples")
print(f"confidence filtering only     : {acc_conf:.2%} "
      f"({keep_conf.sum()}/{N} ≈ {keep_conf.mean():.1%} retained)")
print(f"Confidence + outlier detection  : {acc_combo:.2%} "
      f"({keep_combo.sum()}/{N} ≈ {keep_combo.mean():.1%} retained)")


189/189 [==============================] - 0s 494us/step
Original accuracy             : 82.27% on 6019 samples
confidence filtering only     : 89.41% (4798/6019 ≈ 79.7% retained)
Confidence + outlier detection  : 90.22% (3825/6019 ≈ 63.5% retained)


In [31]:
target_x_train_filtered = target_x_train[keep_combo]
target_pred_train_filtered = y_pred_combo
target_path_train_filtered  = target_path_train[keep_combo]
target_true_train_filtered = y_true_all[keep_combo]

### GMM

In [32]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import accuracy_score

# 1. PCA → 50 dims (speed + regularization)
pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
Z_reduced = pca.fit_transform(Z)                 # (N, 50)

# 2. Softmax probabilities + confidence
probs = classifier.predict(Z)            # shape = (N, num_classes)
conf  = np.max(probs, axis=1)                    # best-class probability

# 3. Build masks: GMM-only, confidence-only, and combined
keep_gmm   = np.zeros(N, dtype=bool)
keep_conf  = np.zeros(N, dtype=bool)
keep_combo = np.zeros(N, dtype=bool)
gmm_contam = 0.20    # fraction of outliers per class
conf_thresh = 0.95   # confidence cutoff

# y_noisy = y_target_class_pred

# 4. Per-class GMM filtering + confidence gating
for c in np.unique(y_noisy):
    idx = np.where(y_noisy == c)[0]
    Zc  = Z_reduced[idx]                        # embeddings for class c

    # fit a single-component GMM
    gmm = GaussianMixture(n_components=1,
                          covariance_type='full',
                          reg_covar=1e-6,
                          random_state=0)
    gmm.fit(Zc)

    # compute log-likelihoods
    log_probs = gmm.score_samples(Zc)           # shape = (n_c,)

    # GMM-only mask (inliers above quantile)
    thresh = np.percentile(log_probs, gmm_contam * 100)
    inliers = log_probs > thresh
    keep_gmm[idx[inliers]] = True

    # confidence-only mask
    conf_mask = conf[idx] >= conf_thresh
    keep_conf[idx[conf_mask]] = True

    # combined mask
    keep_combo[idx[inliers & conf_mask]] = True

# 5. Slice out subsets
y_true_all   = target_y_train.argmax(axis=1)

y_true_conf  = y_true_all[keep_conf]
y_pred_conf  = y_noisy[keep_conf]

y_true_combo = y_true_all[keep_combo]
y_pred_combo = y_noisy[keep_combo]

# 6. Compute & print accuracies
acc_orig = accuracy_score(y_true_all, y_noisy)
acc_conf = accuracy_score(y_true_conf, y_pred_conf)
acc_combo= accuracy_score(y_true_combo, y_pred_combo)

print(f"Original accuracy             : {acc_orig:.2%} on {N} samples")
print(f"confidence filtering only     : {acc_conf:.2%} "
      f"({keep_conf.sum()}/{N} ≈ {keep_conf.mean():.1%} retained)")
print(f"Confidence + outlier detection  : {acc_combo:.2%} "
      f"({keep_combo.sum()}/{N} ≈ {keep_combo.mean():.1%} retained)")


189/189 [==============================] - 0s 509us/step
Original accuracy             : 82.27% on 6019 samples
confidence filtering only     : 89.41% (4798/6019 ≈ 79.7% retained)
Confidence + outlier detection  : 90.48% (3738/6019 ≈ 62.1% retained)


### One class svm 

In [33]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.metrics import accuracy_score

# 1. PCA → 50 dims (speed + regularization)
pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
Z_reduced = pca.fit_transform(Z)                 # (N, 50)

# 2. Softmax probabilities + confidence
probs = classifier.predict(Z)            # shape = (N, num_classes)
conf  = np.max(probs, axis=1)                    # best-class probability

# 3. Build masks: One-Class SVM only, confidence-only, and combined
keep_ocsvm = np.zeros(N, dtype=bool)
keep_conf  = np.zeros(N, dtype=bool)
keep_combo = np.zeros(N, dtype=bool)
svm_nu     = 0.2      # fraction of outliers per class
aic_conf_thresh = 0.95   # confidence cutoff

# 4. Per-class One-Class SVM filtering + confidence gating
y_noisy = y_target_class_pred
for c in np.unique(y_noisy):
    idx = np.where(y_noisy == c)[0]
    Zc  = Z_reduced[idx]                        # embeddings for class c

    # fit One-Class SVM
    ocsvm = OneClassSVM(nu=svm_nu, kernel='rbf', gamma='auto')
    ocsvm.fit(Zc)
    preds = ocsvm.predict(Zc)                   # +1=inlier, -1=outlier
    inliers = preds == 1

    # update One-Class SVM mask
    keep_ocsvm[idx[inliers]] = True

    # update confidence mask
    conf_mask = conf[idx] >= aic_conf_thresh
    keep_conf[idx[conf_mask]] = True

    # update combined mask
    keep_combo[idx[inliers & conf_mask]] = True

# 5. Slice out subsets
y_true_all     = target_y_train.argmax(axis=1)

y_true_conf    = y_true_all[keep_conf]
y_pred_conf    = y_noisy[keep_conf]

y_true_combo   = y_true_all[keep_combo]
y_pred_combo   = y_noisy[keep_combo]

# 6. Compute & print accuracies
acc_orig   = accuracy_score(y_true_all, y_noisy)
acc_conf   = accuracy_score(y_true_conf, y_pred_conf)
acc_combo  = accuracy_score(y_true_combo, y_pred_combo)

print(f"Original accuracy             : {acc_orig:.2%} on {N} samples")
print(f"confidence filtering only     : {acc_conf:.2%} "
      f"({keep_conf.sum()}/{N} ≈ {keep_conf.mean():.1%} retained)")
print(f"Confidence + outlier detection  : {acc_combo:.2%} "
      f"({keep_combo.sum()}/{N} ≈ {keep_combo.mean():.1%} retained)")


189/189 [==============================] - 0s 502us/step
Original accuracy             : 82.27% on 6019 samples
confidence filtering only     : 89.41% (4798/6019 ≈ 79.7% retained)
Confidence + outlier detection  : 90.32% (3731/6019 ≈ 62.0% retained)


### Mahalanobis-distance

In [34]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.covariance import EmpiricalCovariance
from scipy.stats import chi2
from sklearn.metrics import accuracy_score


# 1. PCA → 50 dims (speed + regularization)
pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
Z_reduced = pca.fit_transform(Z)  # (N, 50)

# 2. Softmax probabilities + confidence
probs = classifier.predict(Z)  # shape = (N, num_classes)
conf = np.max(probs, axis=1)    # best-class probability

# 3. Build masks: Mahalanobis-only, confidence-only, and Mahalanobis+confidence
keep_maha = np.zeros(N, dtype=bool)
keep_conf = np.zeros(N, dtype=bool)
keep_combo = np.zeros(N, dtype=bool)
maha_alpha = 0.8    # χ² percentile threshold
conf_thresh = 0.95  # confidence cutoff

# 4. Per-class filtering
# y_noisy = y_target_class_pred
for c in np.unique(y_noisy):
    idx = np.where(y_noisy == c)[0]
    Zc = Z_reduced[idx]

    # Mahalanobis distances
    cov_est = EmpiricalCovariance().fit(Zc)
    m2 = cov_est.mahalanobis(Zc)
    maha_thresh = chi2.ppf(maha_alpha, df=Zc.shape[1])
    maha_mask = m2 < maha_thresh

    # Confidence mask
    conf_mask = conf[idx] >= conf_thresh

    # Update masks
    keep_maha[idx[maha_mask]] = True
    keep_conf[idx[conf_mask]] = True
    keep_combo[idx[maha_mask & conf_mask]] = True

# 5. Slice subsets
y_true_all = target_y_train.argmax(axis=1)

y_true_conf = y_true_all[keep_conf]
y_pred_conf = y_noisy[keep_conf]

y_true_combo = y_true_all[keep_combo]
y_pred_combo = y_noisy[keep_combo]

# 6. Compute & print accuracies
acc_orig = accuracy_score(y_true_all, y_noisy)
acc_conf = accuracy_score(y_true_conf, y_pred_conf)
acc_combo = accuracy_score(y_true_combo, y_pred_combo)

print(f"Original accuracy             : {acc_orig:.2%} on {N} samples")
print(f"confidence filtering only     : {acc_conf:.2%} "
      f"({keep_conf.sum()}/{N} ≈ {keep_conf.mean():.1%} retained)")
print(f"Confidence + outlier detection  : {acc_combo:.2%} "
      f"({keep_combo.sum()}/{N} ≈ {keep_combo.mean():.1%} retained)")


189/189 [==============================] - 0s 493us/step
Original accuracy             : 82.27% on 6019 samples
confidence filtering only     : 89.41% (4798/6019 ≈ 79.7% retained)
Confidence + outlier detection  : 90.65% (3465/6019 ≈ 57.6% retained)


### Isolation forest

In [35]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.ensemble     import IsolationForest
from sklearn.metrics      import accuracy_score

# 2. PCA → 50 dims (speed + regularization)
pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
Z_reduced = pca.fit_transform(Z)                 # (N, 50)


# 4. Softmax probabilities + confidence
probs = classifier.predict(Z) # shape = (N, num_classes)
conf  = np.max(probs, axis=1)                    # best-class probability

# 5. Build two masks:
#    - IsolationForest only
#    - IsolationForest + confidence
keep_iforest = np.zeros(N, dtype=bool)
keep_conf   = np.zeros(N, dtype=bool)
keep_combo   = np.zeros(N, dtype=bool)

iso_contam   = 0.2  # fraction of outliers per class
conf_thresh  = 0.95  # confidence cutoff

for c in np.unique(y_noisy):
    idx = np.where(y_noisy == c)[0]
    Zc  = Z_reduced[idx]                        # embeddings for class c

    # fit IsolationForest on class-c embeddings
    iso = IsolationForest(contamination=iso_contam, random_state=0)
    iso.fit(Zc)
    preds = iso.predict(Zc)                     # +1 inlier, -1 outlier
    inliers = (preds == 1)

    # mark kept for IF only
    keep_iforest[idx[inliers]] = True

    # combine with high-confidence
    conf_mask = conf[idx] > conf_thresh
    keep_conf[idx[conf_mask]] = True

    keep_combo[idx[inliers & conf_mask]] = True

# 6. Slice out subsets
y_true_conf  = y_true_all[keep_conf]
y_pred_conf  = y_noisy[keep_conf]

y_true_combo    = y_true_all[keep_combo]
y_pred_combo    = y_noisy[keep_combo]

# 7. Compute & print accuracies
acc_orig    = accuracy_score(y_true_all, y_noisy)
acc_conf = accuracy_score(y_true_conf, y_pred_conf)
acc_combo   = accuracy_score(y_true_combo, y_pred_combo)

print(f"Original accuracy             : {acc_orig:.2%} on {N} samples")
print(f"confidence filtering only     : {acc_conf:.2%} "
      f"({keep_conf.sum()}/{N} ≈ {keep_conf.mean():.1%} retained)")
print(f"Confidence + outlier detection  : {acc_combo:.2%} "
      f"({keep_combo.sum()}/{N} ≈ {keep_combo.mean():.1%} retained)")


189/189 [==============================] - 0s 497us/step
Original accuracy             : 82.27% on 6019 samples
confidence filtering only     : 89.41% (4798/6019 ≈ 79.7% retained)
Confidence + outlier detection  : 90.48% (3728/6019 ≈ 61.9% retained)


### Save the source and target 

In [36]:
# save_dir = str(REPO_ROOT / "results/stepII_constructed_datasets_scratch/mb24/nov")
# os.makedirs(save_dir, exist_ok=True)

# np.savez_compressed(f'{save_dir}/source_train.npz',
#                     source_path_train=source_path_train, source_y_train = source_y_train.argmax(axis=1))
# np.savez_compressed(f'{save_dir}/source_test.npz',
#                     source_path_test=source_path_test, source_y_test = source_y_test.argmax(axis=1))
# np.savez_compressed(f'{save_dir}/target_train_filtered.npz',
#                     target_path_train_filtered=target_path_train_filtered, target_pred_train_filtered=target_pred_train_filtered, target_true_train_filtered=target_true_train_filtered)
# np.savez_compressed(f'{save_dir}/target_train.npz',
#                     target_path_train=target_path_train,target_y_train=target_y_train.argmax(axis=1))
# np.savez_compressed(f'{save_dir}/target_test.npz',
#                     target_path_test=target_path_test,target_y_test=target_y_test.argmax(axis=1))

# print("Saved Step II constructed datasets to {save_dir}")